In [1]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_curve, mean_squared_error
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool
import optuna
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
    


In [2]:
df = pd.read_parquet('../data/processed/creditcard_clean.parquet')

df.shape

(283726, 31)

# Training

We will create a test dataset first.

Then we will clean outliers and see if this helps for a baseline model to see if we need to proceed with this approach further.

In [4]:
X = df.drop(columns=['Class'])
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Fraud in test: {y_test.sum()}")

Train: 226,980 | Test: 56,746
Fraud in test: 95


In [5]:
df_train = pd.concat([X_train, y_train], axis=1)

In [6]:
sigma_threshold = 5
features = [col for col in df_train.columns if col != 'Class']

df_train_clean = df_train.copy()
for feat in features:
    mean = df_train_clean[feat].mean()
    std = df_train_clean[feat].std()
    df_train_clean = df_train_clean[
        (df_train_clean[feat] < mean + sigma_threshold * std) &
        (df_train_clean[feat] > mean - sigma_threshold * std)
    ]

X_train_clean = df_train_clean.drop(columns=['Class'])
y_train_clean = df_train_clean['Class']
print(f"Train before: {len(df_train):,} | After outlier removal: {len(df_train_clean):,}")
print(f"Fraud before: {df_train['Class'].sum()} | After: {df_train_clean['Class'].sum()}")

Train before: 226,980 | After outlier removal: 212,772
Fraud before: 378 | After: 47


RF baseline on full data

In [19]:
%%time
model_rf = RF(n_estimators=100 , n_jobs = -1, random_state=42)
model_rf.fit(X_train, y_train)

CPU times: user 11min 59s, sys: 23.3 s, total: 12min 22s
Wall time: 2min 43s


RandomForestClassifier(n_jobs=-1, random_state=42)

# Inference

In [20]:
# Make predictions
y_pred_rf = model_rf.predict(X_test)

# Postprocessing

In [21]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.73      0.83        95

    accuracy                           1.00     56746
   macro avg       0.99      0.86      0.92     56746
weighted avg       1.00      1.00      1.00     56746



RF on outlier-removed

# Training

In [22]:
model_rf_clean = RF(n_estimators=100, n_jobs=-1, random_state=42)
model_rf_clean.fit(X_train_clean, y_train_clean)

RandomForestClassifier(n_jobs=-1, random_state=42)

# Inference

In [26]:
# predictions
y_pred_rf_clean = model_rf_clean.predict(X_test)

# Postprocessing

In [27]:
print(classification_report(y_test, y_pred_rf_clean))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.83      0.53      0.65        95

    accuracy                           1.00     56746
   macro avg       0.92      0.76      0.82     56746
weighted avg       1.00      1.00      1.00     56746



In [24]:
print(classification_report(y_test, y_pred_rf_clean))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.73      0.83        95

    accuracy                           1.00     56746
   macro avg       0.99      0.86      0.92     56746
weighted avg       1.00      1.00      1.00     56746



In [25]:
print(f"X_train shape: {X_train.shape}")
print(f"X_train_clean shape: {X_train_clean.shape}")
print(f"Fraud in y_train: {y_train.sum()}")
print(f"Fraud in y_train_clean: {y_train_clean.sum()}")

X_train shape: (226980, 30)
X_train_clean shape: (212772, 30)
Fraud in y_train: 378
Fraud in y_train_clean: 47


**Decision: Do NOT remove outliers.**

Training on outlier-removed data degrades fraud detection significantly:
- Precision: 0.97 → 0.83
- Recall: 0.73 → 0.53
- F1: 0.83 → 0.65

The extreme values in V1–V28 are not noise, they ARE the fraud signal. PCA components shift to extreme ranges specifically for fraudulent transactions. We keep the full dataset.

# Preprocessing

# Oversampling

In [28]:
# Split majority and minority classes
majority = df_train[df_train["Class"] == 0]
minority = df_train[df_train["Class"] == 1]

# Upsample minority class
minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

# Training

In [31]:
# Combine to get balanced training data
train_balanced = pd.concat([majority, minority_upsampled])
X_train_bal = train_balanced.drop("Class", axis=1)
y_train_bal = train_balanced["Class"]

In [32]:
print(f"Original train: {len(df_train):,} (fraud: {df_train['Class'].sum()})")
print(f"Balanced train: {len(train_balanced):,} (fraud: {y_train_bal.sum():,})")

Original train: 226,980 (fraud: 378)
Balanced train: 453,204 (fraud: 226,602)


In [33]:
# Train model
model = RF(n_estimators=100, random_state=42)
model.fit(X_train_bal, y_train_bal)

RandomForestClassifier(random_state=42)

# Inference

In [34]:
# Predict and evaluate on original test set
y_pred_rf_bal = model.predict(X_test)

# Postprocessing

In [35]:
print(classification_report(y_test, y_pred_rf_bal))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.72      0.82        95

    accuracy                           1.00     56746
   macro avg       0.99      0.86      0.91     56746
weighted avg       1.00      1.00      1.00     56746



**Decision: Oversampling does not improve performance.** Oversampling does not improve performance - F1 drops slightly from 0.83 to 0.82. We proceed with the original imbalanced dataset.

# Gradient Boosting

## Training

In [43]:
def objective(trial):
    np.random.seed(42)

    # Hyperparameter search space
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "depth": trial.suggest_int("depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.5, 5.0),
        "iterations": 1000,
        "loss_function": "Logloss",
        "eval_metric": "F1",
        "verbose": 0
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []
    best_iterations = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        x_tr, x_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = CatBoostClassifier(**params, random_seed=42)

        model.fit(
            x_tr, y_tr,
            eval_set=(x_val, y_val),
            early_stopping_rounds=100,
            use_best_model=True
        )

        y_pred = model.predict(x_val)
        f1 = f1_score(y_val, y_pred, average='macro')
        f1_scores.append(f1)
        best_iterations.append(model.get_best_iteration())

    # Save average best iteration across folds
    avg_best_iter = int(np.mean(best_iterations))
    trial.set_user_attr("best_iteration", avg_best_iter)

    return np.mean(f1_scores)

In [44]:
# Step 2: Run the Optuna study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)

# Step 4: Train final model on full x_train with best parameters
best_params = study.best_trial.params

[I 2026-05-27 20:37:06,425] A new study created in memory with name: no-name-3997ae7a-9bf4-4360-aa73-d563f9e501a4
[I 2026-05-27 20:38:06,184] Trial 0 finished with value: 0.9275729058916357 and parameters: {'learning_rate': 0.08116262258099886, 'depth': 8, 'l2_leaf_reg': 3.793972738151323}. Best is trial 0 with value: 0.9275729058916357.
[I 2026-05-27 20:39:23,017] Trial 1 finished with value: 0.9280911840759395 and parameters: {'learning_rate': 0.12374511199743694, 'depth': 3, 'l2_leaf_reg': 1.201975341512912}. Best is trial 1 with value: 0.9280911840759395.
[I 2026-05-27 20:40:25,023] Trial 2 finished with value: 0.9250708661632472 and parameters: {'learning_rate': 0.021035886311957897, 'depth': 8, 'l2_leaf_reg': 3.20501755284444}. Best is trial 1 with value: 0.9280911840759395.
[I 2026-05-27 20:40:59,079] Trial 3 finished with value: 0.9311669995922328 and parameters: {'learning_rate': 0.14453378978124864, 'depth': 3, 'l2_leaf_reg': 4.864594334728975}. Best is trial 3 with value: 0.

In [45]:
best_trial = study.best_trial
print(f"Best F1 score: {best_trial.value:.4f}")
print(f"Best iteration (trees): {best_trial.user_attrs['best_iteration']}")

Best F1 score: 0.9365
Best iteration (trees): 127


In [46]:
best_params.update({
    "iterations": best_trial.user_attrs["best_iteration"],
    "verbose": 0
})

In [47]:
final_model = CatBoostClassifier(**best_params)
final_model.fit(X_train, y_train)

## Inference

In [48]:
# Step 5: Predict on x_test and evaluate
y_pred_gb = final_model.predict(X_test)

# Postprocessing

In [49]:
print(classification_report(y_test, y_pred_gb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.71      0.82        95

    accuracy                           1.00     56746
   macro avg       0.99      0.85      0.91     56746
weighted avg       1.00      1.00      1.00     56746



Summary table

In [50]:
results = pd.DataFrame({
    'Experiment': ['RF Baseline', 'RF + Outlier Removal', 'RF + Oversampling', 'CatBoost + Optuna (30 trials)'],
    'Precision': [0.97, 0.83, 0.97, 0.97],
    'Recall': [0.73, 0.53, 0.72, 0.71],
    'F1': [0.83, 0.65, 0.82, 0.82]
})
print(results.to_string(index=False))

                   Experiment  Precision  Recall   F1
                  RF Baseline       0.97    0.73 0.83
         RF + Outlier Removal       0.83    0.53 0.65
            RF + Oversampling       0.97    0.72 0.82
CatBoost + Optuna (30 trials)       0.97    0.71 0.82


## Conclusions

- We are able to make good predictions with a baseline RF model (F1=0.83, precision=0.97).
- Since we do not know the meaning of V1-V28 (PCA features), feature engineering is limited. However, CatBoost with hyperparameter tuning can extract more signal with additional trials.
- Oversampling did not improve predictions (F1=0.82 vs 0.83 baseline).
- The values which seemed outliers are actually good predictors, removing them drops F1 from 0.83 to 0.65.
- CatBoost with Optuna (30 trials) matches RF but with better scalability and built-in regularization. More trials (100+) would likely outperform RF.
- For deployment, we proceed with CatBoost + Optuna as the production model.

In [51]:
final_model.save_model('../outputs/catboost_model.cbm')
print("Model saved to outputs/catboost_model.cbm")

Model saved to outputs/catboost_model.cbm
